### Architecture

# Gold Fraud Features — v1

## Purpose
Transaction-level historical fraud features based on the authoritative Silver table.

## Grain
One row per Silver transaction.

## Historical Window
30 days.

Historical transactions satisfy:

historical.transaction_timestamp >= current.transaction_timestamp - 30 days

AND

historical.transaction_timestamp < current.transaction_timestamp

## Timestamp Ties
Transactions sharing the same transaction_timestamp are treated as simultaneous.

No secondary ordering is introduced using:
- ingestion_timestamp
- source_file
- row_number()
- arbitrary hashes
- fabricated sequence numbers

## Approved Features

### User
- user_30d_transaction_count
- user_30d_fraud_count
- user_30d_fraud_rate

### Card
- card_30d_transaction_count
- card_30d_fraud_count
- card_30d_fraud_rate

### Merchant
- merchant_30d_transaction_count
- merchant_30d_fraud_count
- merchant_30d_fraud_rate

## NULL / Zero Semantics

Historical counts:
- 0 when no qualifying historical transactions exist.

Historical fraud rates:
- NULL when there are no qualifying historical transactions.

Transactions with NULL transaction_timestamp:
- retained in Gold
- historical features are NULL

## Source
credit_card_project.silver.cleaned_transactions

## Target
credit_card_project.gold.gold_fraud_features


## 1. Source Schema

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = "credit_card_project.silver.cleaned_transactions"
TARGET_TABLE = "credit_card_project.gold.gold_fraud_features"

silver = spark.table(SOURCE_TABLE)

print("Source table:", SOURCE_TABLE)
print("Row count:", f"{silver.count():,}")

silver.select(
    "transaction_timestamp"
).printSchema()


This confirms we're working against the expected 24,386,900-row Silver table and tells us the exact Spark type of transaction_timestamp.

##2. Timestamp distribution and ties

In [0]:
# Basic timestamp population
timestamp_stats = silver.select(
    F.count("transaction_timestamp").alias("non_null_timestamps"),
    F.countDistinct("transaction_timestamp").alias("distinct_timestamps"),
    F.min("transaction_timestamp").alias("min_timestamp"),
    F.max("transaction_timestamp").alias("max_timestamp")
)

timestamp_stats.show(truncate=False)

# Identify exact timestamp ties
tie_check = (
    silver
    .filter(F.col("transaction_timestamp").isNotNull())
    .groupBy("transaction_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

tied_timestamp_values = tie_check.count()

tied_transactions = (
    tie_check
    .agg(F.sum("count").alias("tied_transactions"))
    .first()["tied_transactions"]
)

print(f"Timestamp values with ties: {tied_timestamp_values:,}")
print(f"Transactions participating in ties: {tied_transactions:,}")

print("\nMost frequent tied timestamps:")
display(
    tie_check
    .orderBy(F.desc("count"))
    .limit(20)
)


So all Silver transactions have a non-NULL timestamp, which simplifies the implementation.

More importantly, approximately 21.9M of the 24.4M transactions participate in an exact timestamp tie. That's roughly 90% of the dataset.

24,386,900 total transactions

7,859,684 distinct timestamps

That's a lot of timestamp reuse.


Are most ties 2 transactions, or do some timestamps contain hundreds/thousands?

## 3. Timestamp Tie Analysis

The Silver dataset contains multiple transactions sharing identical
transaction timestamps.

These transactions are treated as simultaneous for historical-feature
purposes. No secondary ordering is introduced.

This section examines the size and distribution of timestamp ties before
production feature calculation.


In [0]:
tie_distribution = (
    silver
    .filter(F.col("transaction_timestamp").isNotNull())
    .groupBy("transaction_timestamp")
    .count()
)

print("Timestamp tie-size distribution:")

tie_distribution.select(
    F.min("count").alias("min_transactions_per_timestamp"),
    F.max("count").alias("max_transactions_per_timestamp"),
    F.avg("count").alias("avg_transactions_per_timestamp"),
    F.expr("percentile_approx(count, 0.50)").alias("median_transactions_per_timestamp"),
    F.expr("percentile_approx(count, 0.90)").alias("p90_transactions_per_timestamp"),
    F.expr("percentile_approx(count, 0.99)").alias("p99_transactions_per_timestamp")
).show()


In [0]:
print("Most heavily tied timestamps:")

display(
    tie_distribution
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(20)
)


Stats:

- 24,386,900 / 24,386,900 transactions have a timestamp.
- There are 7,859,684 distinct timestamps.
- 5,400,009 timestamp values are tied.
- 21,927,225 transactions participate in ties.
- Tie sizes range from 2 to 20 transactions.
- Median tie size is 2.
- 90% of timestamp groups have at most 6 transactions.
- 99% have at most 10 transactions.

So the decision to treat identical timestamps as simultaneous is very appropriate.

Also, because the maximum tie is only 20 transactions, there isn't some unusual giant batch timestamp we need to design around.

## 4. Temporal Window Semantics Test

The production historical window is:

historical_timestamp >= current_timestamp - 30 days

AND

historical_timestamp < current_timestamp

Therefore:

- exactly 30 days before the current transaction is included
- transactions at the current timestamp are excluded
- transactions after the current timestamp are excluded
- transactions older than 30 days are excluded
- timestamp ties are treated as simultaneous


In [0]:
# Controlled test data
test_data = [
    ("T_CURRENT",  "2020-02-28 12:00:00", 0),
    ("T_TIE",      "2020-02-28 12:00:00", 1),
    ("T_30D",      "2020-01-29 12:00:00", 1),
    ("T_30D_PLUS", "2020-01-29 11:59:59", 1),
    ("T_RECENT",   "2020-02-20 12:00:00", 1),
]

test_df = (
    spark.createDataFrame(
        test_data,
        ["transaction_id", "transaction_timestamp", "is_fraud"]
    )
    .withColumn(
        "transaction_timestamp",
        F.to_timestamp("transaction_timestamp")
    )
    .withColumn(
        "_timestamp_seconds",
        F.col("transaction_timestamp").cast("long")
    )
)

test_df.orderBy("transaction_timestamp").show(truncate=False)


In [0]:
TEST_30D_SECONDS = 30 * 24 * 60 * 60

test_window = (
    Window
    .orderBy(F.col("_timestamp_seconds"))
    .rangeBetween(-TEST_30D_SECONDS, -1)
)

test_result = (
    test_df
    .withColumn(
        "historical_count",
        F.count(F.lit(1)).over(test_window)
    )
    .withColumn(
        "historical_fraud_count",
        F.sum(
            F.when(F.col("is_fraud") == 1, 1).otherwise(0)
        ).over(test_window)
    )
    .orderBy("transaction_timestamp", "transaction_id")
)

display(
    test_result.select(
        "transaction_id",
        "transaction_timestamp",
        "is_fraud",
        "historical_count",
        "historical_fraud_count"
    )
)


## 5. Source Key and Temporal Readiness Checks

Before production feature calculation, validate the business keys used by
the three historical feature grains:

- user
- user + card
- merchant

All Silver transactions currently have non-NULL transaction timestamps.


In [0]:
# Check NULLs in the business keys used by the historical features
key_nulls = silver.select(
    F.sum(F.col("user_id").isNull().cast("int")).alias("null_user_id"),
    F.sum(F.col("card_id").isNull().cast("int")).alias("null_card_id"),
    F.sum(F.col("merchant_id").isNull().cast("int")).alias("null_merchant_id"),
    F.sum(F.col("transaction_timestamp").isNull().cast("int")).alias("null_transaction_timestamp")
)

key_nulls.show()


In [0]:
# Check the cardinality of the historical feature grains
silver.select(
    F.countDistinct("user_id").alias("distinct_users"),
    F.countDistinct(
        F.struct("user_id", "card_id")
    ).alias("distinct_user_card_pairs"),
    F.countDistinct("merchant_id").alias("distinct_merchants")
).show()


- Users:                    2,000 => user_id is safe for the user history
- User/card combinations:  6,139 => (user_id, card_id) is the card history grain.
- Merchants:              100,343 => merchant_id is the merchant history grain.

Every transaction has a usable timestamp.


## 6. Production Feature Preparation

The Gold table retains the transaction-level context required for fraud
analysis plus the approved 30-day historical fraud features.

Historical features are calculated from the authoritative Silver source.

No transaction ordering key is introduced.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = "credit_card_project.silver.cleaned_transactions"
TARGET_TABLE = "credit_card_project.gold.gold_fraud_features"

DAYS_30_SECONDS = 30 * 24 * 60 * 60

tx = (
    silver
    .select(
        "user_id",
        "card_id",
        "transaction_timestamp",
        "amount",
        "merchant_id",
        "merchant_category_code",
        "is_fraud",
        "is_online",
        "payment_method"
    )
    .withColumn(
        "_timestamp_seconds",
        F.col("transaction_timestamp").cast("long") # numeric value that Spark can use with rangeBetween()
    )
)

print(f"Prepared transaction rows: {tx.count():,}")


## 7. User 30-Day Historical Fraud Features

For each transaction, calculate historical activity for the same user over
the preceding 30 days.

Window definition:

historical_timestamp >= current_timestamp - 30 days
AND
historical_timestamp < current_timestamp

Transactions sharing the current timestamp are excluded from each other's
history.

No artificial transaction ordering is introduced.


In [0]:
user_window = (
    Window
    .partitionBy("user_id")
    .orderBy(F.col("_timestamp_seconds"))
    .rangeBetween(-DAYS_30_SECONDS, -1)
)

tx = (
    tx
    .withColumn(
        "user_30d_transaction_count",
        F.count(F.lit(1)).over(user_window)
    )
    .withColumn(
        "user_30d_fraud_count",
        F.sum(
            F.when(F.col("is_fraud") == True, 1).otherwise(0)
        ).over(user_window)
    )
)

tx = (
    tx
    .withColumn(
        "user_30d_transaction_count",
        F.coalesce(
            F.col("user_30d_transaction_count"),
            F.lit(0)
        ).cast("long")
    )
    .withColumn(
        "user_30d_fraud_count",
        F.coalesce(
            F.col("user_30d_fraud_count"),
            F.lit(0)
        ).cast("long")
    )
    .withColumn(
        "user_30d_fraud_rate",
        F.when(
            F.col("user_30d_transaction_count") > 0,
            F.col("user_30d_fraud_count") /
            F.col("user_30d_transaction_count")
        )
    )
)


Let's first validate that the user features look sensible:

In [0]:
user_feature_summary = tx.select(
    F.count("*").alias("rows"),
    F.sum(
        F.when(F.col("user_30d_transaction_count") == 0, 1).otherwise(0)
    ).alias("rows_with_no_user_history"),
    F.min("user_30d_transaction_count").alias("min_user_30d_transactions"),
    F.max("user_30d_transaction_count").alias("max_user_30d_transactions"),
    F.min("user_30d_fraud_count").alias("min_user_30d_fraud"),
    F.max("user_30d_fraud_count").alias("max_user_30d_fraud"),
    F.min("user_30d_fraud_rate").alias("min_user_30d_fraud_rate"),
    F.max("user_30d_fraud_rate").alias("max_user_30d_fraud_rate")
)

user_feature_summary.show(truncate=False)


In [0]:
display(
    tx.select(
        "user_id",
        "card_id",
        "transaction_timestamp",
        "is_fraud",
        "user_30d_transaction_count",
        "user_30d_fraud_count",
        "user_30d_fraud_rate"
    )
    .orderBy("user_id", "transaction_timestamp")
    .limit(30)
)


## 8. Card 30-Day Historical Fraud Features

For each transaction, calculate historical activity for the same
user/card combination over the preceding 30 days.

Historical transactions satisfy:

historical_timestamp >= current_timestamp - 30 days

AND

historical_timestamp < current_timestamp

Transactions sharing the current timestamp are treated as simultaneous and
cannot contribute to one another's historical features.


In [0]:
card_window = (
    Window
    .partitionBy("user_id", "card_id")
    .orderBy(F.col("_timestamp_seconds"))
    .rangeBetween(-DAYS_30_SECONDS, -1)
)

tx = (
    tx
    .withColumn(
        "card_30d_transaction_count",
        F.count(F.lit(1)).over(card_window)
    )
    .withColumn(
        "card_30d_fraud_count",
        F.sum(
            F.when(F.col("is_fraud") == True, 1).otherwise(0)
        ).over(card_window)
    )
    .withColumn(
        "card_30d_transaction_count",
        F.coalesce(
            F.col("card_30d_transaction_count"),
            F.lit(0)
        ).cast("long")
    )
    .withColumn(
        "card_30d_fraud_count",
        F.coalesce(
            F.col("card_30d_fraud_count"),
            F.lit(0)
        ).cast("long")
    )
    .withColumn(
        "card_30d_fraud_rate",
        F.when(
            F.col("card_30d_transaction_count") > 0,
            F.col("card_30d_fraud_count") /
            F.col("card_30d_transaction_count")
        )
    )
)


In [0]:
card_feature_summary = tx.select(
    F.count("*").alias("rows"),
    F.sum(
        F.when(F.col("card_30d_transaction_count") == 0, 1).otherwise(0)
    ).alias("rows_with_no_card_history"),
    F.min("card_30d_transaction_count").alias("min_card_30d_transactions"),
    F.max("card_30d_transaction_count").alias("max_card_30d_transactions"),
    F.min("card_30d_fraud_count").alias("min_card_30d_fraud"),
    F.max("card_30d_fraud_count").alias("max_card_30d_fraud"),
    F.min("card_30d_fraud_rate").alias("min_card_30d_fraud_rate"),
    F.max("card_30d_fraud_rate").alias("max_card_30d_fraud_rate")
)

card_feature_summary.show(truncate=False)


In [0]:
display(
    tx.select(
        "user_id",
        "card_id",
        "transaction_timestamp",
        "is_fraud",
        "card_30d_transaction_count",
        "card_30d_fraud_count",
        "card_30d_fraud_rate"
    )
    .orderBy("user_id", "card_id", "transaction_timestamp")
    .limit(30)
)


The card-level features are also behaving as expected:

- Rows:                         24,386,900
- No 30-day card history:            7,845
- Min historical transactions:           0
- Max historical transactions:         467
- Min historical fraud count:            0
- Max historical fraud count:           29
- Min fraud rate:                      0.0
- Max fraud rate:                      1.0


## 9. Merchant 30-Day Historical Fraud Features

For each transaction, calculate historical activity for the same merchant
over the preceding 30 days.

Historical transactions satisfy:

historical_timestamp >= current_timestamp - 30 days

AND

historical_timestamp < current_timestamp

Transactions sharing the current timestamp are treated as simultaneous and
cannot contribute to one another's historical features.

merchant_id is used as the business key. It is not interpreted as a
physical location.


In [0]:
merchant_window = (
    Window
    .partitionBy("merchant_id")
    .orderBy(F.col("_timestamp_seconds"))
    .rangeBetween(-DAYS_30_SECONDS, -1)
)

tx = (
    tx
    .withColumn(
        "merchant_30d_transaction_count",
        F.count(F.lit(1)).over(merchant_window)
    )
    .withColumn(
        "merchant_30d_fraud_count",
        F.sum(
            F.when(F.col("is_fraud") == True, 1).otherwise(0)
        ).over(merchant_window)
    )
    .withColumn(
        "merchant_30d_transaction_count",
        F.coalesce(
            F.col("merchant_30d_transaction_count"),
            F.lit(0)
        ).cast("long")
    )
    .withColumn(
        "merchant_30d_fraud_count",
        F.coalesce(
            F.col("merchant_30d_fraud_count"),
            F.lit(0)
        ).cast("long")
    )
    .withColumn(
        "merchant_30d_fraud_rate",
        F.when(
            F.col("merchant_30d_transaction_count") > 0,
            F.col("merchant_30d_fraud_count") /
            F.col("merchant_30d_transaction_count")
        )
    )
)


In [0]:
merchant_feature_summary = tx.select(
    F.count("*").alias("rows"),
    F.sum(
        F.when(
            F.col("merchant_30d_transaction_count") == 0,
            1
        ).otherwise(0)
    ).alias("rows_with_no_merchant_history"),
    F.min("merchant_30d_transaction_count").alias(
        "min_merchant_30d_transactions"
    ),
    F.max("merchant_30d_transaction_count").alias(
        "max_merchant_30d_transactions"
    ),
    F.min("merchant_30d_fraud_count").alias(
        "min_merchant_30d_fraud"
    ),
    F.max("merchant_30d_fraud_count").alias(
        "max_merchant_30d_fraud"
    ),
    F.min("merchant_30d_fraud_rate").alias(
        "min_merchant_30d_fraud_rate"
    ),
    F.max("merchant_30d_fraud_rate").alias(
        "max_merchant_30d_fraud_rate"
    )
)

merchant_feature_summary.show(truncate=False)


In [0]:
display(
    tx.select(
        "user_id",
        "card_id",
        "merchant_id",
        "transaction_timestamp",
        "is_fraud",
        "merchant_30d_transaction_count",
        "merchant_30d_fraud_count",
        "merchant_30d_fraud_rate"
    )
    .orderBy("merchant_id", "transaction_timestamp")
    .limit(30)
)


The merchant features also look consistent:
- Rows:                         24,386,900
- No 30-day merchant history:    1,140,527
- Min historical transactions:           0
- Max historical transactions:       7,990
- Min historical fraud count:            0
- Max historical fraud count:           71
- Fraud rate range:                0.0 → 1.0


## 10. Timestamp Precision Validation

The production range window uses a numeric representation of
transaction_timestamp.

Before finalizing the transformation, verify that the Silver timestamps do
not contain sub-second precision that would be lost by casting to seconds.


In [0]:
timestamp_precision_check = (
    silver
    .filter(F.col("transaction_timestamp").isNotNull())
    .select(
        F.max(
            F.date_format(
                "transaction_timestamp",
                "SSS"
            ).cast("int")
        ).alias("max_millisecond_component")
    )
)

timestamp_precision_check.show()


In [0]:
timestamp_seconds_check = (
    silver
    .filter(F.col("transaction_timestamp").isNotNull())
    .select(
        F.max(
            F.second("transaction_timestamp")
        ).alias("max_second_component")
    )
)

timestamp_seconds_check.show()


## 11. Final Gold Projection

The Gold table contains one row per Silver transaction.

It retains the transaction context needed to interpret the fraud features
and the approved 30-day historical features at three business grains:

- user
- user + card
- merchant

The temporary timestamp calculation column is not persisted.


In [0]:
gold = tx.select(
    "user_id",
    "card_id",
    "transaction_timestamp",
    "amount",
    "merchant_id",
    "merchant_category_code",
    "is_fraud",
    "is_online",
    "payment_method",

    "user_30d_transaction_count",
    "user_30d_fraud_count",
    "user_30d_fraud_rate",

    "card_30d_transaction_count",
    "card_30d_fraud_count",
    "card_30d_fraud_rate",

    "merchant_30d_transaction_count",
    "merchant_30d_fraud_count",
    "merchant_30d_fraud_rate"
)

gold.printSchema()


In [0]:
gold_count = gold.count()

assert gold_count == 24_386_900, (
    f"Row-count mismatch: Silver={24_386_900:,}, Gold={gold_count:,}"
)

print(f"PASS: Gold population = {gold_count:,}")


## 12. Validate the feature ranges

In [0]:
rate_columns = [
    "user_30d_fraud_rate",
    "card_30d_fraud_rate",
    "merchant_30d_fraud_rate"
]

for c in rate_columns:
    invalid = gold.filter(
        (F.col(c) < 0) | (F.col(c) > 1)
    ).count()

    assert invalid == 0, (
        f"{c}: {invalid} invalid values"
    )

    print(f"PASS: {c} range")


In [0]:
count_columns = [
    "user_30d_transaction_count",
    "user_30d_fraud_count",
    "card_30d_transaction_count",
    "card_30d_fraud_count",
    "merchant_30d_transaction_count",
    "merchant_30d_fraud_count"
]

for c in count_columns:
    invalid = gold.filter(
        F.col(c) < 0
    ).count()

    assert invalid == 0, (
        f"{c}: {invalid} negative values"
    )

    print(f"PASS: {c} non-negative")


## 13. Fraud reconciliation
We also need to make sure the transaction-level Gold object hasn't altered the underlying fraud population.

In [0]:
silver_fraud = (
    silver
    .filter(F.col("is_fraud") == True)
    .count()
)

gold_fraud = (
    gold
    .filter(F.col("is_fraud") == True)
    .count()
)

print(f"Silver fraud count: {silver_fraud:,}")
print(f"Gold fraud count:   {gold_fraud:,}")

assert silver_fraud == 29_757
assert gold_fraud == silver_fraud

print("PASS: fraud reconciliation")


## 14. Historical Leakage Validation

Historical features must use only transactions with a strictly earlier
transaction_timestamp within the preceding 30 days.

Transactions sharing the current transaction_timestamp must not contribute
to one another's historical features.

This validation compares the production window result against an
independently calculated strict-prior result on a targeted sample.


In [0]:
tie_timestamps = (
    silver
    .filter(F.col("transaction_timestamp").isNotNull())
    .groupBy("transaction_timestamp")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Timestamp values with ties: {tie_timestamps.count():,}")


In [0]:
sample_timestamps = (
    tie_timestamps
    .orderBy(F.desc("count"), "transaction_timestamp")
    .limit(10)
    .select("transaction_timestamp")
)

display(sample_timestamps)


In [0]:
sample_tx = (
    silver
    .join(
        sample_timestamps,
        on="transaction_timestamp",
        how="inner"
    )
    .select(
        "user_id",
        "card_id",
        "merchant_id",
        "transaction_timestamp",
        "is_fraud"
    )
)

display(
    sample_tx
    .orderBy("transaction_timestamp", "user_id", "card_id")
)


This gives us the actual tied transactions we're testing.
Now independently calculate the expected user history.

This deliberately doesn't reuse the production window expression.

In [0]:
sample_with_expected = (
    sample_tx.alias("current")
    .join(
        silver.select(
            "user_id",
            "transaction_timestamp",
            "is_fraud"
        ).alias("history"),
        on=(
            (F.col("current.user_id") == F.col("history.user_id")) &
            (F.col("history.transaction_timestamp") >=
             F.col("current.transaction_timestamp") -
             F.expr("INTERVAL 30 DAYS")) &
            (F.col("history.transaction_timestamp") <
             F.col("current.transaction_timestamp"))
        ),
        how="left"
    )
    .groupBy(
        "current.user_id",
        "current.card_id",
        "current.merchant_id",
        "current.transaction_timestamp",
        "current.is_fraud"
    )
    .agg(
        F.count("history.transaction_timestamp").alias(
            "expected_user_30d_transaction_count"
        ),
        F.sum(
            F.when(F.col("history.is_fraud") == True, 1)
             .otherwise(0)
        ).alias(
            "expected_user_30d_fraud_count"
        )
    )
)


Now compare this independently calculated result against our production feature:

In [0]:
user_leakage_check = (
    sample_with_expected.alias("expected")
    .join(
        gold.select(
            "user_id",
            "card_id",
            "transaction_timestamp",
            "user_30d_transaction_count",
            "user_30d_fraud_count"
        ).alias("actual"),
        on=[
            "user_id",
            "card_id",
            "transaction_timestamp"
        ],
        how="inner"
    )
    .filter(
        (F.col("expected_user_30d_transaction_count") !=
         F.col("actual.user_30d_transaction_count")) |
        (
            F.coalesce(
                F.col("expected_user_30d_fraud_count"),
                F.lit(0)
            ) !=
            F.col("actual.user_30d_fraud_count")
        )
    )
)

mismatches = user_leakage_check.count()

print(f"User historical-feature mismatches: {mismatches:,}")

assert mismatches == 0, (
    f"Found {mismatches:,} user historical-feature mismatches"
)

print("PASS: strict user timestamp semantics")


This test is valuable (given: only against 10 selected tied timestamps) as we are independently expressing the rule as:

history.timestamp >= current.timestamp - 30 days
AND
history.timestamp < current.timestamp


## 15. Card and Merchant Historical Leakage Validation

Validate the production card-level and merchant-level features against
independently calculated strict-prior results.

The validation uses the same tied-timestamp sample and does not introduce
any artificial transaction ordering.


In [0]:
card_expected = (
    sample_tx.alias("current")
    .join(
        silver.select(
            "user_id",
            "card_id",
            "transaction_timestamp",
            "is_fraud"
        ).alias("history"),
        on=(
            (F.col("current.user_id") == F.col("history.user_id")) &
            (F.col("current.card_id") == F.col("history.card_id")) &
            (F.col("history.transaction_timestamp") >=
             F.col("current.transaction_timestamp") -
             F.expr("INTERVAL 30 DAYS")) &
            (F.col("history.transaction_timestamp") <
             F.col("current.transaction_timestamp"))
        ),
        how="left"
    )
    .groupBy(
        "current.user_id",
        "current.card_id",
        "current.merchant_id",
        "current.transaction_timestamp",
        "current.is_fraud"
    )
    .agg(
        F.count("history.transaction_timestamp").alias(
            "expected_card_30d_transaction_count"
        ),
        F.sum(
            F.when(F.col("history.is_fraud") == True, 1)
             .otherwise(0)
        ).alias(
            "expected_card_30d_fraud_count"
        )
    )
)


In [0]:
card_leakage_check = (
    card_expected.alias("expected")
    .join(
        gold.select(
            "user_id",
            "card_id",
            "transaction_timestamp",
            "card_30d_transaction_count",
            "card_30d_fraud_count"
        ).alias("actual"),
        on=[
            "user_id",
            "card_id",
            "transaction_timestamp"
        ],
        how="inner"
    )
    .filter(
        (F.col("expected_card_30d_transaction_count") !=
         F.col("actual.card_30d_transaction_count")) |
        (
            F.coalesce(
                F.col("expected_card_30d_fraud_count"),
                F.lit(0)
            ) !=
            F.col("actual.card_30d_fraud_count")
        )
    )
)

card_mismatches = card_leakage_check.count()

print(f"Card historical-feature mismatches: {card_mismatches:,}")

assert card_mismatches == 0, (
    f"Found {card_mismatches:,} card historical-feature mismatches"
)

print("PASS: strict card timestamp semantics")


In [0]:
merchant_expected = (
    sample_tx.alias("current")
    .join(
        silver.select(
            "merchant_id",
            "transaction_timestamp",
            "is_fraud"
        ).alias("history"),
        on=(
            (F.col("current.merchant_id") == F.col("history.merchant_id")) &
            (F.col("history.transaction_timestamp") >=
             F.col("current.transaction_timestamp") -
             F.expr("INTERVAL 30 DAYS")) &
            (F.col("history.transaction_timestamp") <
             F.col("current.transaction_timestamp"))
        ),
        how="left"
    )
    .groupBy(
        "current.user_id",
        "current.card_id",
        "current.merchant_id",
        "current.transaction_timestamp",
        "current.is_fraud"
    )
    .agg(
        F.count("history.transaction_timestamp").alias(
            "expected_merchant_30d_transaction_count"
        ),
        F.sum(
            F.when(F.col("history.is_fraud") == True, 1)
             .otherwise(0)
        ).alias(
            "expected_merchant_30d_fraud_count"
        )
    )
)


In [0]:
merchant_leakage_check = (
    merchant_expected.alias("expected")
    .join(
        gold.select(
            "user_id",
            "card_id",
            "merchant_id",
            "transaction_timestamp",
            "merchant_30d_transaction_count",
            "merchant_30d_fraud_count"
        ).alias("actual"),
        on=[
            "user_id",
            "card_id",
            "merchant_id",
            "transaction_timestamp"
        ],
        how="inner"
    )
    .filter(
        (F.col("expected_merchant_30d_transaction_count") !=
         F.col("actual.merchant_30d_transaction_count")) |
        (
            F.coalesce(
                F.col("expected_merchant_30d_fraud_count"),
                F.lit(0)
            ) !=
            F.col("actual.merchant_30d_fraud_count")
        )
    )
)

merchant_mismatches = merchant_leakage_check.count()

print(f"Merchant historical-feature mismatches: {merchant_mismatches:,}")

assert merchant_mismatches == 0, (
    f"Found {merchant_mismatches:,} merchant historical-feature mismatches"
)

print("PASS: strict merchant timestamp semantics")


## 16. Final Pre-Write Structural Validation

Confirm the Gold projection has exactly the approved transaction context and
historical feature columns.

The temporary `_timestamp_seconds` calculation column must not be persisted.

The Gold grain remains one row per Silver transaction.


In [0]:
expected_columns = [
    "user_id",
    "card_id",
    "transaction_timestamp",
    "amount",
    "merchant_id",
    "merchant_category_code",
    "is_fraud",
    "is_online",
    "payment_method",

    "user_30d_transaction_count",
    "user_30d_fraud_count",
    "user_30d_fraud_rate",

    "card_30d_transaction_count",
    "card_30d_fraud_count",
    "card_30d_fraud_rate",

    "merchant_30d_transaction_count",
    "merchant_30d_fraud_count",
    "merchant_30d_fraud_rate"
]

actual_columns = gold.columns

print(f"Expected columns: {len(expected_columns)}")
print(f"Actual columns:   {len(actual_columns)}")

assert actual_columns == expected_columns, (
    f"Schema mismatch.\n"
    f"Expected: {expected_columns}\n"
    f"Actual:   {actual_columns}"
)

assert "_timestamp_seconds" not in actual_columns

print("PASS: Gold schema")
print("PASS: temporary timestamp column excluded")


Then let's verify the key transaction fields haven't unexpectedly changed through the transformation:

In [0]:
source_summary = silver.select(
    F.count("*").alias("row_count"),
    F.sum("amount").alias("total_amount"),
    F.sum(
        F.when(F.col("is_fraud") == True, 1).otherwise(0)
    ).alias("fraud_count")
).collect()[0]

gold_summary = gold.select(
    F.count("*").alias("row_count"),
    F.sum("amount").alias("total_amount"),
    F.sum(
        F.when(F.col("is_fraud") == True, 1).otherwise(0)
    ).alias("fraud_count")
).collect()[0]

print("Silver:")
print(f"  rows:        {source_summary['row_count']:,}")
print(f"  total amount:{source_summary['total_amount']}")
print(f"  fraud count: {source_summary['fraud_count']:,}")

print("Gold:")
print(f"  rows:        {gold_summary['row_count']:,}")
print(f"  total amount:{gold_summary['total_amount']}")
print(f"  fraud count: {gold_summary['fraud_count']:,}")

assert source_summary["row_count"] == gold_summary["row_count"]
assert source_summary["total_amount"] == gold_summary["total_amount"]
assert source_summary["fraud_count"] == gold_summary["fraud_count"]

print("PASS: transaction population reconciliation")
print("PASS: signed amount reconciliation")
print("PASS: fraud reconciliation")


Why we're doing the amount check:

This is another useful guardrail from our previous Gold work. The fraud-feature transformation should not alter the underlying transaction population or amounts.

## 17. Persist Gold Fraud Features

Persist the validated transaction-level fraud feature DataFrame as a Delta
table.

The table is written only after all pre-write validation checks have passed.


In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Persisted: {TARGET_TABLE}")


## 18. Reload and Validate Persisted Gold Table

Reload the Gold table from the catalog and validate the persisted Delta
object independently of the in-memory DataFrame.


In [0]:
gold_persisted = spark.table(TARGET_TABLE)

print("Persisted schema:")
gold_persisted.printSchema()


In [0]:
persisted_count = gold_persisted.count()

print(f"Persisted Gold rows: {persisted_count:,}")

assert persisted_count == 24_386_900, (
    f"Persisted row count mismatch: {persisted_count:,}"
)

print("PASS: persisted row count")


In [0]:
persisted_summary = gold_persisted.select(
    F.count("*").alias("row_count"),
    F.sum("amount").alias("total_amount"),
    F.sum(
        F.when(F.col("is_fraud") == True, 1).otherwise(0)
    ).alias("fraud_count")
).collect()[0]

print(f"Rows:         {persisted_summary['row_count']:,}")
print(f"Total amount: {persisted_summary['total_amount']}")
print(f"Fraud count:  {persisted_summary['fraud_count']:,}")

assert persisted_summary["row_count"] == 24_386_900
assert persisted_summary["total_amount"] == source_summary["total_amount"]
assert persisted_summary["fraud_count"] == 29_757

print("PASS: persisted row count")
print("PASS: persisted signed amount reconciliation")
print("PASS: persisted fraud reconciliation")


## 20. Persisted feature validation

Now make sure the values survived persistence correctly:

In [0]:
for c in rate_columns:
    invalid = gold_persisted.filter(
        (F.col(c) < 0) | (F.col(c) > 1)
    ).count()

    assert invalid == 0, (
        f"{c}: {invalid} invalid persisted rates"
    )

    print(f"PASS: persisted {c} range")


In [0]:
for c in count_columns:
    invalid = gold_persisted.filter(
        F.col(c) < 0
    ).count()

    assert invalid == 0, (
        f"{c}: {invalid} negative persisted values"
    )

    print(f"PASS: persisted {c} non-negative")


21. Confirm the final schema

In [0]:
persisted_columns = gold_persisted.columns

assert persisted_columns == expected_columns, (
    f"Persisted schema mismatch.\n"
    f"Expected: {expected_columns}\n"
    f"Actual:   {persisted_columns}"
)

assert "_timestamp_seconds" not in persisted_columns

print(f"Persisted column count: {len(persisted_columns)}")
print("PASS: persisted schema")
print("PASS: temporary timestamp column absent")


## 22. Final engineering review

In [0]:
print("=== GOLD FRAUD FEATURES — FINAL REVIEW ===")

print(f"Table: {TARGET_TABLE}")
print(f"Rows: {gold_persisted.count():,}")
print(f"Columns: {len(gold_persisted.columns)}")
print(f"Fraud rows: {gold_persisted.filter(F.col('is_fraud') == True).count():,}")

print("\nApproved historical features:")
print("- User:     30-day transaction count, fraud count, fraud rate")
print("- Card:     30-day transaction count, fraud count, fraud rate")
print("- Merchant: 30-day transaction count, fraud count, fraud rate")

print("\nTemporal semantics:")
print("- Window: 30 days")
print("- Lower boundary: inclusive")
print("- Upper boundary: strict < current timestamp")
print("- Same-timestamp transactions: excluded from each other's history")
print("- Artificial transaction ordering: none")

print("\nValidation:")
print("PASS: source population reconciliation")
print("PASS: fraud reconciliation")
print("PASS: signed amount reconciliation")
print("PASS: feature range validation")
print("PASS: timestamp leakage validation")
print("PASS: persisted-table validation")
print("PASS: schema validation")

print("\nSTATUS: GOLD FRAUD FEATURES COMPLETE")
